In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data'
TODAY    = pd.Timestamp.today().normalize()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

subs      = pd.read_csv(DATA_DIR / 'subscriptions.csv',
                        parse_dates=['signup_date', 'churn_date'])
events    = pd.read_csv(DATA_DIR / 'product_events.csv',
                        parse_dates=['event_date'])
leads     = pd.read_csv(DATA_DIR / 'leads.csv',
                        parse_dates=['created_date', 'converted_date'])
campaigns = pd.read_csv(DATA_DIR / 'email_campaigns.csv',
                        parse_dates=['send_date'])
sends     = pd.read_csv(DATA_DIR / 'email_sends.csv',
                        parse_dates=['send_date'])

print("All tables loaded ")
print(f"  {'Table':<22} {'Rows':>8}   {'Columns':>8}")
print(f"  {'-'*42}")
for name, df in [
    ('subscriptions',   subs),
    ('product_events',  events),
    ('leads',           leads),
    ('email_campaigns', campaigns),
    ('email_sends',     sends),
]:
    print(f"  {name:<22} {len(df):>8,}   {len(df.columns):>8}")

All tables loaded 
  Table                      Rows    Columns
  ------------------------------------------
  subscriptions               500         16
  product_events          212,110          6
  leads                     1,700         13
  email_campaigns              34          6
  email_sends               2,962          8


In [2]:
print("TABLE 1: subscriptions.csv")
print("Source of truth for revenue and churn.")
print("Every other table links back here via user_id.")


print("\n Schema ")
schema = {
    'user_id':          'Primary key — shared across all tables',
    'company_name':     'B2B company name',
    'email':            'Contact email',
    'plan':             'starter / growth / pro / enterprise',
    'mrr':              'Monthly recurring revenue in USD',
    'status':           'active / churned',
    'signup_date':      'When the user subscribed',
    'churn_date':       'Cancellation date (null if active)',
    'churn_reason':     'Why they left (null if active)',
    'payment_failures': 'Count of failed billing attempts',
    'country':          'ISO 2-letter country code',
    'industry':         'Company vertical',
}
for col, desc in schema.items():
    print(f"  {col:<20} {desc}")

print("\n Sample rows ")
print(subs[['user_id','plan','mrr','status','signup_date',
            'churn_date','churn_reason','payment_failures']].head(6).to_string(index=False))

print("\n Plan distribution ")
plan_stats = subs.groupby('plan').agg(
    users       = ('user_id',  'count'),
    churned     = ('status',   lambda x: (x == 'churned').sum()),
    active_mrr  = ('mrr',      lambda x: x[subs.loc[x.index, 'status'] == 'active'].sum()),
).reindex(['starter','growth','pro','enterprise'])
plan_stats['churn_rate'] = (
    plan_stats['churned'] / plan_stats['users'] * 100
).round(1).astype(str) + '%'
plan_stats['active_mrr'] = plan_stats['active_mrr'].apply(lambda x: f'${x:,.0f}')
print(plan_stats.to_string())

print("\n Churn reasons ")
print(subs['churn_reason'].value_counts().to_string())

print("\n Key design decisions ")
print("  1. Churn rate varies by plan tier (enterprise ~3%, starter ~37%)")
print("  2. Enterprise has 180-day minimum tenure before churn is possible")
print("  3. Payment failures are correlated with churn (not random)")
print("  4. Churn reasons are plan-aware (starter=too_expensive, enterprise=contract_ended)")
print("  5. Signup dates weighted toward recent months to simulate growth")

TABLE 1: subscriptions.csv
Source of truth for revenue and churn.
Every other table links back here via user_id.

 Schema 
  user_id              Primary key — shared across all tables
  company_name         B2B company name
  email                Contact email
  plan                 starter / growth / pro / enterprise
  mrr                  Monthly recurring revenue in USD
  status               active / churned
  signup_date          When the user subscribed
  churn_date           Cancellation date (null if active)
  churn_reason         Why they left (null if active)
  payment_failures     Count of failed billing attempts
  country              ISO 2-letter country code
  industry             Company vertical

 Sample rows 
 user_id    plan  mrr  status signup_date churn_date        churn_reason  payment_failures
usr_0001 starter   49  active  2026-03-25        NaT                 NaN                 0
usr_0002  growth  149  active  2026-04-13        NaT                 NaN         

In [3]:
# Product events overview 

print("Top event types:")
print(events['event_name'].value_counts())

events_with_plan = events.merge(
    subs[['user_id', 'plan', 'status']],
    on='user_id',
    how='left'
)

activity = (
    events_with_plan.groupby('plan')
    .agg(
        total_events=('event_id', 'count'),
        users=('user_id', 'nunique'),
    )
    .reindex(['starter', 'growth', 'pro', 'enterprise'])
)

activity['events_per_user'] = (
    activity['total_events'] / activity['users']
).round(0)

print("\nActivity by plan:")
print(activity)

avg_events = (
    events_with_plan.groupby('status')
    .apply(lambda x: x.groupby('user_id').size().mean())
    .round(0)
)

print("\nAvg events per user:")
print(avg_events)

print("\nSample events:")
print(
    events_with_plan[
        ['event_name', 'event_date', 'platform', 'plan', 'status']
    ].head()
)

Top event types:
event_name
login                     34905
sso_login                 17661
report_created            14677
dashboard_view            12978
integration_connected     10941
automation_created        10721
export_csv                10367
team_member_invited       10049
scheduled_report_set       9320
bulk_export                9016
webhook_created            8556
api_called                 8059
custom_dashboard_built     7948
role_permission_set        7123
audit_log_viewed           7011
filter_applied             6427
data_governance_view       6101
comment_added              4672
report_viewed              4558
api_key_generated          4033
settings_viewed            2206
chart_customised           1855
share_link_created         1371
help_viewed                 826
logout                      729
Name: count, dtype: int64

Activity by plan:
            total_events  users  events_per_user
plan                                            
starter            13468    1

In [4]:
#  Leads overview 

stage_order = ['new', 'qualified', 'demo_booked', 'converted']

print("Funnel stages:")
print(leads['stage'].value_counts().reindex(stage_order))

conv = (
    leads.groupby('source')
    .apply(lambda x: pd.Series({
        'total': len(x),
        'converted': (x['stage'] == 'converted').sum(),
        'conv_rate': round((x['stage'] == 'converted').mean() * 100, 1)
    }))
    .sort_values('conv_rate', ascending=False)
)

print("\nConversion by source:")
print(conv)

converted = leads[leads['stage'] == 'converted']
linked = converted['user_id'].isin(subs['user_id'])

print(f"\nConverted leads linked to subscriptions: {linked.mean()*100:.1f}%")

print("\nSample rows:")
print(
    leads[
        ['source', 'stage', 'created_date', 'converted_date']
    ].head()
)

Funnel stages:
stage
new            235
qualified      491
demo_booked    474
converted      500
Name: count, dtype: int64

Conversion by source:
            total  converted  conv_rate
source                                 
cold_call   183.0       67.0       36.6
paid_ad     130.0       43.0       33.1
cold_email  481.0      140.0       29.1
linkedin    398.0      111.0       27.9
inbound     294.0       81.0       27.6
referral    214.0       58.0       27.1

Converted leads linked to subscriptions: 100.0%

Sample rows:
       source        stage created_date converted_date
0   cold_call    converted   2025-09-27     2025-10-08
1   cold_call  demo_booked   2026-04-11            NaT
2    linkedin  demo_booked   2025-08-28            NaT
3  cold_email    qualified   2026-02-26            NaT
4    linkedin          new   2026-02-26            NaT


In [5]:
#  Email datasets overview 

print("Campaign types:")
print(campaigns['campaign_type'].value_counts())

perf = (
    sends.merge(
        campaigns[['campaign_id', 'campaign_type']],
        on='campaign_id'
    )
    .groupby('campaign_type')
    .agg(
        sends=('send_id', 'count'),
        open_rate=('opened', 'mean'),
        reply_rate=('replied', 'mean'),
        unsub_rate=('unsubscribed', 'mean'),
    )
    .round(3)
    .sort_values('reply_rate', ascending=False)
)

print("\nCampaign performance:")
print(perf)

print("\nSample campaigns:")
print(
    campaigns[
        ['campaign_name', 'campaign_type', 'segment', 'send_date']
    ].head()
)

Campaign types:
campaign_type
newsletter       12
re_engagement     8
upsell            6
winback           4
promo             4
Name: count, dtype: int64

Campaign performance:
               sends  open_rate  reply_rate  unsub_rate
campaign_type                                          
promo            634      0.394       0.021       0.008
newsletter      1455      0.364       0.008       0.005
upsell           656      0.387       0.008       0.006
winback          196      0.097       0.005       0.026
re_engagement     21      0.143       0.000       0.048

Sample campaigns:
                   campaign_name  campaign_type         segment  send_date
0    Re-engagement #1 — Apr 2025  re_engagement    low_activity 2025-04-28
1    Upgrade Offer #1 — May 2025         upsell  starter_growth 2025-05-11
2    Re-engagement #2 — May 2025  re_engagement    low_activity 2025-05-17
3  Monthly Newsletter — May 2025     newsletter      all_active 2025-05-31
4    Re-engagement #3 — Jun 2025  r

In [6]:
#  Join validation 

checks = {
    'events → subs':
        events['user_id'].isin(subs['user_id']).mean(),

    'leads(conv) → subs':
        leads.loc[leads['stage'] == 'converted', 'user_id']
        .isin(subs['user_id']).mean(),

    'sends → subs':
        sends['user_id'].isin(subs['user_id']).mean(),

    'sends → campaigns':
        sends['campaign_id'].isin(campaigns['campaign_id']).mean(),
}

join_summary = (
    pd.Series(checks)
    .mul(100)
    .round(1)
    .rename('match_%')
)

print(join_summary)

# Sample joined dataset
joined = events.merge(
    subs[['user_id', 'plan', 'status', 'mrr']],
    on='user_id',
    how='left'
)

joined[
    ['event_name', 'event_date', 'plan', 'status', 'mrr']
].head()

events → subs         100.0
leads(conv) → subs    100.0
sends → subs          100.0
sends → campaigns     100.0
Name: match_%, dtype: float64


,event_name,event_date,plan,status,mrr
0,login,2026-03-25,starter,active,49
1,report_viewed,2026-03-25,starter,active,49
2,dashboard_view,2026-03-25,starter,active,49
3,login,2026-03-26,starter,active,49
4,dashboard_view,2026-03-26,starter,active,49
